# 2 Training and regularization

We will use the breast-cancer dataset to compare learning rates and dropout, inspect learning curves, and evaluate a selected model. All data preparation and training code is included below.

In [ ]:
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(2)

seed_everything(42)

## 2.1 Reuse the workflow
We use exactly the same deterministic split as notebook 01. All numeric inputs are standardized using training-only statistics. There is one output logit and no final sigmoid inside the model.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from copy import deepcopy
from torch import nn
import pandas as pd

def cancer_splits(seed=42):
    """60/20/20 stratified split; fit preprocessing on training data only.

    Relabel explicitly: 1 = malignant, 0 = benign (opposite sklearn's coding).
    The returned test arrays should be opened only after model selection.
    """
    source = load_breast_cancer()
    x = source.data.astype(np.float32)
    y = (source.target == 0).astype(np.float32)
    idx = np.arange(len(y))
    train, remainder = train_test_split(idx, test_size=0.4, stratify=y, random_state=seed)
    val, test = train_test_split(remainder, test_size=0.5, stratify=y[remainder], random_state=seed)
    scaler = StandardScaler().fit(x[train])
    result = {"scaler": scaler, "feature_names": source.feature_names}
    for name, rows in [("train", train), ("val", val), ("test", test)]:
        result[name] = (scaler.transform(x[rows]).astype(np.float32), y[rows])
        result[name + "_ids"] = rows
    return result

def loader(x, y, batch_size=32, shuffle=False, seed=42):
    return DataLoader(
        TensorDataset(torch.as_tensor(x, dtype=torch.float32), torch.as_tensor(y, dtype=torch.float32)),
        batch_size=batch_size, shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed), num_workers=0,
    )

def fit_binary(model, train_loader, val_loader, epochs=30, lr=0.001, weight_decay=0.0):
    """CPU training for small exercises. Return independent best-validation weights.

    Models return ONE LOGIT per example. The loss includes the sigmoid.
    Loss means are weighted by sample count, including the last short batch.
    """
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_loss = float("inf")
    best_weights = deepcopy(model.state_dict())
    history = []
    for epoch in range(epochs):
        values = {}
        for phase, batches in [("train", train_loader), ("val", val_loader)]:
            model.train(phase == "train")
            total_loss, count = 0.0, 0
            with torch.set_grad_enabled(phase == "train"):
                for x, y in batches:
                    if phase == "train":
                        optimizer.zero_grad()
                    logits = model(x).squeeze(-1)
                    loss = loss_fn(logits, y)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                    total_loss += loss.item() * len(y)
                    count += len(y)
            values[phase + "_loss"] = total_loss / count
        if values["val_loss"] < best_loss:
            best_loss = values["val_loss"]
            best_weights = deepcopy(model.state_dict())
        history.append({"epoch": epoch + 1, **values})
    model.load_state_dict(best_weights)
    model.eval()
    return pd.DataFrame(history)

In [ ]:
import numpy as np, pandas as pd, torch
from torch import nn
import matplotlib.pyplot as plt
splits = cancer_splits()
x_train, y_train = splits["train"]
x_val, y_val = splits["val"]
val_loader = loader(x_val, y_val)

def make_model(width=32, dropout=0.0):
    return nn.Sequential(nn.Linear(30, width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, 1))

def run_experiment(lr=0.001, dropout=0.0, width=32, epochs=40):
    seed_everything(42)
    model = make_model(width, dropout)
    history = fit_binary(model, loader(x_train, y_train, shuffle=True), val_loader,
                         lr=lr, epochs=epochs)
    return model, history

## 2.2 Learning rate: how far do we move?
An epoch is one pass through the training data. We hold the split, initialization, architecture and number of epochs fixed while comparing learning rates. The training curves help distinguish slow progress from unstable updates.

*Assignment: Change one thing*

Run with 0.0001 and 0.01. Predict which will learn faster before running. Compare the curves; do not use the test set to decide.

In [ ]:
learning_rates = [0.0001]  # Add 0.01 to compare two settings.
rate_runs = {}
for lr in learning_rates:
    model, history = run_experiment(lr=lr)
    rate_runs[lr] = (model, history)
    plt.plot(history.epoch, history.val_loss, label=f"lr={lr}")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.legend()
plt.show()

*Write your answer here.*

## 2.3 Overfitting: practice gets better, new examples do not
An expressive model can fit details specific to its training samples. A widening gap between training and validation loss can suggest this, although small validation sets produce noisy curves.

**Dropout** randomly zeros some hidden activations during training and rescales the retained activations. It is disabled in evaluation mode. It may help generalization, but it is not guaranteed to improve every dataset.

*Assignment: Compare dropout*

Keep width, learning rate and epochs fixed. Compare dropout 0 and 0.3. Record whether you see clearer evidence of overfitting and whether dropout helps.

In [ ]:
dropout_rates = [0.0]  # Add 0.3.
dropout_runs = {}
fig, ax = plt.subplots(figsize=(7, 4))
for dropout in dropout_rates:
    model, history = run_experiment(lr=0.01, dropout=dropout, width=64, epochs=60)
    dropout_runs[dropout] = (model, history)
    ax.plot(history.epoch, history.train_loss, linestyle="--", label=f"train, dropout={dropout}")
    ax.plot(history.epoch, history.val_loss, label=f"validation, dropout={dropout}")
ax.set(xlabel="Epoch", ylabel="Loss")
ax.legend()
plt.show()

*Write your answer here.*

## 2.4 Batch normalization is a different operation
Batch normalization rescales intermediate activations using batch statistics during training and running statistics during evaluation, then applies learned scale and shift. It can change optimization; it is not a replacement for a proper split or input preprocessing.

For this small introductory dataset we leave it out of the model. The important habit is calling `model.train()` during learning and `model.eval()` during evaluation. We will meet **layer normalization** in transformers; it normalizes within an example rather than across the batch.

In [ ]:
example_dropout = nn.Dropout(0.5)
ones = torch.ones(10)
example_dropout.train()
print("Training mode:", example_dropout(ones))
example_dropout.eval()
print("Evaluation mode:", example_dropout(ones))

## 2.5 Select on validation, then open the test set once
We select the run and epoch with the lowest validation loss among the experiments above. More experiments mean more chances to overfit validation; a fresh test set still matters.

For classification we use a threshold of 0.5 fixed in advance. Changing a threshold is also a model choice: do that using training/validation, not the test set.

In [ ]:
candidates = {f"lr={lr}": run for lr, run in rate_runs.items()}
candidates.update({f"dropout={rate}, width=64": run for rate, run in dropout_runs.items()})
summary = pd.DataFrame([{"setting": name, "best_validation_loss": history.val_loss.min()}
                        for name, (model, history) in candidates.items()]).sort_values("best_validation_loss")
display(summary)
selected_name = summary.iloc[0].setting
selected_model = candidates[selected_name][0]
print("Selected before looking at test labels:", selected_name)

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

@torch.no_grad()
def probabilities(model, x):
    model.eval()
    return torch.sigmoid(model(torch.as_tensor(x, dtype=torch.float32)).squeeze(-1)).numpy()

def binary_scores(y, probability):
    predicted = np.asarray(probability) >= 0.5
    return {
        "accuracy": accuracy_score(y, predicted),
        "balanced_accuracy": balanced_accuracy_score(y, predicted),
        "ROC_AUC": roc_auc_score(y, probability),
    }

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
x_test, y_test = splits["test"]
test_probabilities = probabilities(selected_model, x_test)
display(pd.Series(binary_scores(y_test, test_probabilities), name="Held-out test"))
print(classification_report(y_test, test_probabilities >= 0.5,
                            target_names=["benign", "malignant"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, test_probabilities >= 0.5,
                                        display_labels=["benign", "malignant"], cmap="Blues")
plt.show()

*Assignment: Interpret the result*

Which mistakes appear in the confusion matrix? Can this test establish that the model works on samples from another hospital? After seeing these results, what data would you need to evaluate further changes fairly?

*Write your answer here.*

*Optional assignment: The width-zero baseline*

Write a model factory where `hidden_layers=0` actually returns a single linear layer. Why would always creating the first hidden layer invalidate a sweep containing zero?

In [ ]:
def architecture(hidden_layers=0, width=32):
    return nn.Linear(30, 1)  # Extend this to support 1 or more hidden layers.
print(architecture(0))

*Write your answer here.*